In [3]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import pandas as pd
import os
import plotly.express as px
load_dotenv()

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

df_conn_test = pd.read_sql("SELECT * FROM abfragen LIMIT 5", engine)
df_conn_test.head()

,id,tankstellen_id,diesel,e5,e10,isopen,retrieval_time,retrieval_date
0,97,a7cdd9cf-b467-4aac-8eab-d662f082511e,1.689,1.759,1.699,True,11:47,2026-02-15
1,98,e102dc2a-a9db-4a70-adfa-4abd42ad7dcb,1.699,1.759,1.699,True,11:47,2026-02-15
2,99,4f4eadab-2845-4504-8acc-50a19bec8583,1.699,1.759,1.699,True,11:47,2026-02-15
3,100,4d726ab3-6ad3-45c1-a652-362ae6d62c6b,1.699,1.759,1.699,True,11:47,2026-02-15
4,101,6903db69-dd2c-4a8c-b657-caec419c7a32,1.699,1.759,1.699,True,11:47,2026-02-15


In [4]:
df = pd.read_sql("SELECT * FROM abfragen", engine)

# Grundüberblick
print(f"Gesamte Datensätze: {len(df)}")
print(f"Zeitraum: {df['retrieval_date'].min()} bis {df['retrieval_date'].max()}")
print(f"Anzahl eindeutige Tankstellen: {df['tankstellen_id'].nunique()}")
print(f"Anzahl eindeutige Tage: {df['retrieval_date'].nunique()}")

# Datensätze pro Tag
print(f"\nDatensätze pro Tag:")
print(df.groupby('retrieval_date').size())

Gesamte Datensätze: 188640
Zeitraum: 2026-02-15 bis 2026-03-28
Anzahl eindeutige Tankstellen: 96
Anzahl eindeutige Tage: 42

Datensätze pro Tag:
retrieval_date
2026-02-15    2496
2026-02-16    4608
2026-02-17    4608
2026-02-18    4608
2026-02-19    4608
2026-02-20    4608
2026-02-21    4608
2026-02-22    4608
2026-02-23    4608
2026-02-24    4608
2026-02-25    4608
2026-02-26    4608
2026-02-27    4608
2026-02-28    4608
2026-03-01    4608
2026-03-02    4608
2026-03-03    4608
2026-03-04    4608
2026-03-05    4608
2026-03-06    4608
2026-03-07    4608
2026-03-08    4608
2026-03-09    4608
2026-03-10    4608
2026-03-11    4608
2026-03-12    4608
2026-03-13    4608
2026-03-14    4608
2026-03-15    4608
2026-03-16    4608
2026-03-17    4608
2026-03-18    4608
2026-03-19    4608
2026-03-20    4608
2026-03-21    4608
2026-03-22    4608
2026-03-23    4608
2026-03-24    4608
2026-03-25    4608
2026-03-26    4608
2026-03-27    4608
2026-03-28    1824
dtype: int64


In [5]:
# Fehlende Werte
print("Fehlende Werte pro Spalte:")
print(df.isnull().sum())

print(f"\nDatentypen:")
print(df.dtypes)

Fehlende Werte pro Spalte:
id                  0
tankstellen_id      0
diesel             62
e5                302
e10                22
isopen              0
retrieval_time      0
retrieval_date      0
dtype: int64

Datentypen:
id                  int64
tankstellen_id        str
diesel            float64
e5                float64
e10               float64
isopen               bool
retrieval_time        str
retrieval_date     object
dtype: object


In [6]:
df_stations = pd.read_sql("SELECT * FROM tankstellen", engine)
df_stations.head()

,id,name,brand,street,place,lat,lng,dist,housenumber,postcode
0,a7cdd9cf-b467-4aac-8eab-d662f082511e,STUTTGART - KRIEGSBERGSTRASSE 55 A,AGIP ENI,Kriegsbergstrasse,Stuttgart,48.782290,9.171490,0.8,55 A,70174
1,e102dc2a-a9db-4a70-adfa-4abd42ad7dcb,AVIA S-Bebelstr.,AVIA,Bebelstr.,Stuttgart,48.774800,9.158100,1.0,9,70176
2,4f4eadab-2845-4504-8acc-50a19bec8583,Shell Stuttgart Rosenbergplatz 7,Shell,Rosenbergplatz,Stuttgart,48.778509,9.155544,1.3,7,70193
3,4d726ab3-6ad3-45c1-a652-362ae6d62c6b,Esso Station,ESSO,Immenhofer Str. 48,Stuttgart,48.762952,9.176572,1.4,,70180
4,6903db69-dd2c-4a8c-b657-caec419c7a32,Shell Stuttgart Karl-Kloss-Str. 18,Shell,Karl-Kloss-Str.,Stuttgart,48.759586,9.159391,1.9,18,70199


In [32]:

color_map = {
    "Shell":        "#FFD700",  # Gelb
    "ARAL":         "#0066CC",  # Blau
    "ESSO":         "#FF0000",  # Rot
    "AGIP ENI":     "#009900",  # Grün
    "JET":          "#FF6600",  # Orange
    "AVIA":         "#CC00CC",  # Lila
}

# Alle anderen Marken grau
andere_marken = [b for b in df_stations['brand'].unique() if b not in color_map]
for marke in andere_marken:
    color_map[marke] = "#AAAAAA"

fig = px.scatter_map(
    df_stations,
    lat="lat",
    lon="lng",
    hover_name="name",
    hover_data=["brand"],
    zoom=10,
    #color="brand",
    #color_discrete_map=color_map,
    map_style="open-street-map",
)


fig.update_traces(marker=dict(size=15, opacity=0.7))
fig.show()
# Karte als HTML speichern (Karte geht nicht als PNG)
fig.write_html("assets/karte.html")


In [8]:
# Durchschnittspreis pro Messpunkt (retrieval_date + retrieval_time)
df_avg = df.groupby('retrieval_date')[['diesel', 'e5', 'e10']].mean().reset_index()

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(x=df_avg['retrieval_date'], y=df_avg['diesel'], name='Diesel', mode='lines'))
fig.add_trace(go.Scatter(x=df_avg['retrieval_date'], y=df_avg['e5'], name='E5', mode='lines'))
fig.add_trace(go.Scatter(x=df_avg['retrieval_date'], y=df_avg['e10'], name='E10', mode='lines'))

fig.update_layout(
    title='Durchschnittlicher Spritpreis im Raum Stuttgart',
    xaxis_title='Datum',
    yaxis_title='Preis (€)',
    hovermode='x unified'
)

fig.update_xaxes(
    dtick="D1",
    tickformat="%d.%m",
    tickangle=45
)


fig.add_vline(
    x=pd.Timestamp("2026-02-28").timestamp() * 1000,
    line_dash="dash",
    line_color="red",
    annotation_text="28.02.",
    annotation_position="top"
)

fig.show()

# Preisverlauf speichern
fig.write_image("assets/preisverlauf.png", width=1668, height=450)


In [9]:
brand_counts = df_stations['brand'].value_counts().reset_index()
brand_counts.columns = ['brand', 'anzahl']
print(brand_counts)

fig = px.bar(
    brand_counts,
    x='brand',
    y='anzahl',
    title='Anzahl Tankstellen pro Marke im Raum Stuttgart',
    labels={'brand': 'Marke', 'anzahl': 'Anzahl'},
    text='anzahl'
)

fig.update_layout(xaxis_tickangle=45)

fig.write_image("assets/brand_counts.png", width=1668, height=450)

                               brand  anzahl
0                              Shell      18
1                               ARAL      16
2                               ESSO      12
3                           AGIP ENI       9
4                                JET       9
5                               AVIA       8
6                        AVIA XPress       3
7                      TotalEnergies       3
8                                RAN       3
9              Supermarkt-Tankstelle       3
10                                SB       1
11                  Freie Tankstelle       1
12           Mr. Wash Autoservice AG       1
13                            Sprint       1
14                            Access       1
15                 SCHARR-Tankstelle       1
16                     ORLEN Express       1
17                               BFT       1
18                               HEM       1
19                     SB Tankstelle       1
20                  freie Tankstelle       1
21  MTB Au

In [10]:
from datetime import date

mask = (df['retrieval_date'] >= date(2026, 2, 16)) & (df['retrieval_date'] <= date(2026, 2, 28))
df_filtered = df[mask].copy()
print(df_filtered['retrieval_date'].min())
print(df_filtered['retrieval_date'].max())
print(len(df_filtered))

df_filtered['hour'] = pd.to_datetime(df_filtered['retrieval_time'], format='%H:%M').dt.hour

df_hourly = df_filtered.groupby('hour')[['diesel', 'e5', 'e10']].mean().reset_index()

fig1 = px.line(
    df_hourly,
    x='hour',
    y=['diesel', 'e5', 'e10'],
    title='Durchschnittlicher Preisverlauf über den Tag (16.02 - 28.02)',
    labels={'hour': 'Uhrzeit', 'value': 'Preis (€)', 'variable': 'Kraftstoff'}
)
fig1.update_xaxes(dtick=1)
fig1.show()
fig.write_image("assets/avg_price_over_day.png", width=1668, height=450)

2026-02-16
2026-02-28
59904


In [11]:
df_7uhr = df_filtered[df_filtered['hour'] == 7]

df_melted = df_7uhr.melt(
    value_vars=['diesel', 'e5', 'e10'],
    var_name='kraftstoff',
    value_name='preis'
)

fig = px.box(
    df_melted,
    y='preis',
    title='Preisschwankung um 7 Uhr (16.02 - 28.02)',
    labels={'preis': 'Preis (€)'},
    points='outliers'
)
fig.show()

In [21]:
# Durchschnittlicher E5 Preis pro Tankstelle pro Tag
df_daily = df_joined.groupby(['retrieval_date', 'name', 'brand'])['e5'].mean().reset_index()

# Pro Tag die 3 günstigsten und 3 teuersten
results = []
for date, group in df_daily.groupby('retrieval_date'):
    sorted_group = group.sort_values('e5')
    cheapest = sorted_group.head(3).assign(kategorie='günstigste 3')
    expensive = sorted_group.tail(3).assign(kategorie='teuerste 3')
    results.append(pd.concat([cheapest, expensive]))

df_ranked = pd.concat(results).reset_index(drop=True)
df_ranked['label'] = df_ranked['name'] + ' (' + df_ranked['brand'] + ')'

print(df_ranked.columns.tolist())
print(df_ranked.head())

fig = px.scatter(
    df_ranked,
    x='retrieval_date',
    y='e5',
    color='kategorie',
    hover_name='label',
    color_discrete_map={
        'günstigste 3': 'green',
        'teuerste 3': 'red'
    },
    title='Günstigste und teuerste Tankstellen pro Tag (E5)',
    labels={
        'retrieval_date': 'Datum',
        'e5': 'E5 Preis (€)',
        'kategorie': 'Kategorie'
    }
)

fig.update_traces(marker=dict(size=10))

fig.update_xaxes(
    dtick="D1",
    tickformat="%d.%m",
    tickangle=45
)

fig.show()

['retrieval_date', 'name', 'brand', 'e5', 'kategorie', 'label']
  retrieval_date                                               name  \
0     2026-02-15  Supermarkt-Tankstelle GERLINGEN WEILIMDORFER S...   
1     2026-02-15                                 AVIA S-Rappachstr.   
2     2026-02-15                                      MTS Waschpark   
3     2026-02-15                                    Esso Tankstelle   
4     2026-02-15                                    AVIA Tankstelle   

                   brand        e5     kategorie  \
0  Supermarkt-Tankstelle  1.704769  günstigste 3   
1                   AVIA  1.712462  günstigste 3   
2          SB Tankstelle  1.719000  günstigste 3   
3                   ESSO  1.793692    teuerste 3   
4                   AVIA  1.822923    teuerste 3   

                                               label  
0  Supermarkt-Tankstelle GERLINGEN WEILIMDORFER S...  
1                          AVIA S-Rappachstr. (AVIA)  
2                      MTS Wasc

In [ ]:
df_avg_station = df_joined.groupby(['name', 'brand'])['e5'].mean().reset_index()

# Koordinaten aus tankstellen Tabelle dazujoinen
df_avg_station = df_avg_station.merge(
    df_stations[['name', 'lat', 'lng']],
    on='name',
    how='left'
)

df_avg_station = df_avg_station.sort_values('e5').reset_index(drop=True)
df_avg_station['e5'] = df_avg_station['e5'].round(3)
df_avg_station.columns = ['Name', 'Marke', 'Ø E5 Preis (€)', 'Lat', 'Lng']

df_avg_station


,Name,Marke,Ø E5 Preis (€),Lat,Lng
0,MTS Waschpark,SB Tankstelle,1.895,48.735304,9.287473
1,Supermarkt-Tankstelle ESSLINGEN WANNENRAIN 30,Supermarkt-Tankstelle,1.911,48.742440,9.267730
2,SCHARR-Tankstelle,SCHARR-Tankstelle,1.915,48.719587,9.124684
3,ORLEN Express Tankstelle,ORLEN Express,1.918,48.757178,9.273507
4,Supermarkt-Tankstelle KORNTAL-MUENCHINGEN MOTO...,Supermarkt-Tankstelle,1.930,48.827900,9.108680
...,...,...,...,...,...
91,Aral Tankstelle,ARAL,1.996,48.723200,9.159887
92,Aral Tankstelle,ARAL,1.996,48.850590,9.105471
93,Aral Tankstelle,ARAL,1.996,48.822903,9.069422
94,STUTTGART - KRIEGSBERGSTRASSE 55 A,AGIP ENI,1.996,48.782290,9.171490


In [29]:
color_map = {
    top3_expensive[0]: '#8B0000',  # dunkelrot
    top3_expensive[1]: '#006400',  # dunkelgrün
    top3_expensive[2]: '#00008B',  # dunkelblau
    top3_cheap[0]: '#FF6B6B',      # hellrot
    top3_cheap[1]: '#90EE90',      # hellgrün
    top3_cheap[2]: '#87CEEB',      # hellblau
}

fig_trend = px.line(
    df_selected_daily,
    x='retrieval_date',
    y='e5',
    color='name',
    title='Preisentwicklung E5 - Top 3 günstigste vs. teuerste Tankstellen',
    labels={
        'retrieval_date': 'Datum',
        'e5': 'E5 Preis (€)',
        'name': 'Tankstelle'
    },
    color_discrete_map=color_map
)

fig_trend.update_xaxes(
    dtick="D1",
    tickformat="%d.%m",
    tickangle=45
)

fig_trend.show()

In [30]:
# Top 5 günstigste und teuerste Tankstellen
top5_cheap = df_avg_station.head(5)['Name'].tolist()
top5_expensive = df_avg_station.tail(5)['Name'].tolist()
selected = top5_cheap + top5_expensive

# Preisentwicklung dieser 10 Tankstellen filtern
df_selected = df_joined[df_joined['name'].isin(selected)]
df_selected_daily = df_selected.groupby(['retrieval_date', 'name', 'brand'])['e5'].mean().reset_index()

# Kategorie hinzufügen
df_selected_daily['kategorie'] = df_selected_daily['name'].apply(
    lambda x: 'günstigste 5' if x in top5_cheap else 'teuerste 5'
)

# Farben
color_map = {
    top5_expensive[0]: '#8B0000',
    top5_expensive[1]: '#006400',
    top5_expensive[2]: '#00008B',
    top5_expensive[3]: '#8B4513',
    top5_expensive[4]: '#4B0082',
    top5_cheap[0]: '#FF6B6B',
    top5_cheap[1]: '#90EE90',
    top5_cheap[2]: '#87CEEB',
    top5_cheap[3]: '#FFB347',
    top5_cheap[4]: '#DDA0DD',
}

fig_trend = px.line(
    df_selected_daily,
    x='retrieval_date',
    y='e5',
    color='name',
    title='Preisentwicklung E5 - Top 5 günstigste vs. teuerste Tankstellen',
    labels={
        'retrieval_date': 'Datum',
        'e5': 'E5 Preis (€)',
        'name': 'Tankstelle'
    },
    color_discrete_map=color_map
)

fig_trend.update_xaxes(
    dtick="D1",
    tickformat="%d.%m",
    tickangle=45
)

fig_trend.show()

In [36]:
df_selected_daily

,retrieval_date,name,brand,e5,kategorie
0,2026-02-15,Aral Tankstelle,ARAL,1.792317,teuerste 5
1,2026-02-15,MTS Waschpark,SB Tankstelle,1.719000,günstigste 5
2,2026-02-15,ORLEN Express Tankstelle,ORLEN Express,1.727846,günstigste 5
3,2026-02-15,SCHARR-Tankstelle,SCHARR-Tankstelle,1.741308,günstigste 5
4,2026-02-15,STUTTGART - KRIEGSBERGSTRASSE 55 A,AGIP ENI,1.789000,teuerste 5
...,...,...,...,...,...
331,2026-03-28,SCHARR-Tankstelle,SCHARR-Tankstelle,2.052684,günstigste 5
332,2026-03-28,STUTTGART - KRIEGSBERGSTRASSE 55 A,AGIP ENI,2.152684,teuerste 5
333,2026-03-28,Supermarkt-Tankstelle ESSLINGEN WANNENRAIN 30,Supermarkt-Tankstelle,2.066368,günstigste 5
334,2026-03-28,Supermarkt-Tankstelle KORNTAL-MUENCHINGEN MOTO...,Supermarkt-Tankstelle,2.034789,günstigste 5


In [ ]:



fig = px.scatter_map(
    df_selected_daily,
    lat="lat",
    lon="lng",
    hover_name="name",
    hover_data=["brand"],
    zoom=10,
    #color="brand",
    #color_discrete_map=color_map,
    map_style=color_map
)


fig.update_traces(marker=dict(size=15, opacity=0.7))
fig.show()
# Karte als HTML speichern (Karte geht nicht als PNG)
fig.write_html("assets/karte.html")


ValueError: Value of 'lat' is not the name of a column in 'data_frame'. Expected one of ['retrieval_date', 'name', 'brand', 'e5', 'kategorie'] but received: lat